# SmartProposal: AI-Powered Grant Writing for Nonprofits Using Gemini AI

## Section 1: Introduction
## Introduction

Nonprofits often face a common and frustrating challenge: they know what impact they want to create, but they lack the time, resources, or expertise to write strong, funder-aligned grant proposals.

This project demonstrates how **Generative AI (Gemini 2.0 Flash)** can streamline that entire process — from understanding an NGO’s mission to generating a high-quality proposal that’s tailored to the most relevant Canadian grant opportunities.

### Objective:
To build an AI-powered pipeline that:
- Understands an NGO's mission
- Matches it with suitable Canadian grants
- Generates a high-quality, personalized grant proposal
- Produces a structured action plan with timelines and alignment scoring

This solution uses **real-world nonprofit challenges** to demonstrate the potential of GenAI in reducing the overhead of funding applications and expanding access to opportunity.

---

## GenAI Capabilities Demonstrated


- **Retrieval Augmented Generation (RAG)** – Match NGO goals with grant descriptions using keyword/semantic logic
- **Few-Shot Prompting** – Guide proposal generation with quality examples
- **Structured Output (JSON)** – Output formatted results ready for applications
- **GenAI Evaluation** – Gemini assigns fit scores based on mission alignment
- **Prompt Chaining** – Multi-stage generation: analysis → writing → structuring



## Section 2: Setup and Configuration

In this section, we set up Gemini 2.0 Flash via the `google-generativeai` library and load the API key using Kaggle Secrets.

We also prepare the environment for prompt generation and text analysis.


In [1]:
# Section 2: Setup and Configuration
# Install and import required libraries
!pip install -q -U google-generativeai fpdf ipywidgets

import pandas as pd
import numpy as np
import os
import json
import google.generativeai as genai
from google.generativeai.types import GenerationConfig
from kaggle_secrets import UserSecretsClient
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, FileLink
from fpdf import FPDF



  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.4/175.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.0/214.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 50.2 MB/s eta 0:00:00:00:01


In [2]:
# Gemini API Setup Using Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai

# Load your secret key securely
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

# Configure Gemini properly
genai.configure(api_key=GOOGLE_API_KEY)

# Choose your model
model = genai.GenerativeModel(model_name="gemini-2.0-flash")

print(" Gemini 2.0 Flash is ready.")


 Gemini 2.0 Flash is ready.


## Section 3: Load the Canadian Grants Dataset

This section dynamically loads **real-time grant data** from the Government of Canada’s Open Data Portal using the CKAN API.

Instead of relying on a static CSV, the system:

- Uses the **CKAN Search API** to discover the latest dataset matching "grants contributions"
- Retrieves the most recent **CSV resource URL**
- Loads the dataset into memory for processing

Only **nonprofit grants** are retained for this use case.

Each grant entry includes:
- Name of the grant recipient
- Focus area (e.g., innovation, education)
- Region (Province/Territory)
- Funding amount (cleaned and formatted)
- Description and program purpose (merged for matching)

This ensures my Gemini-powered matcher is working with the **latest available data**, and can adapt to new grant opportunities as they are published.

In [3]:
# SECTION 2: Dataset Discovery from Open Canada CKAN API

def fetch_latest_csv_url_from_open_canada(query="grants contributions"):
    search_url = f"https://open.canada.ca/data/api/3/action/package_search?q={query}&rows=1"
    response = requests.get(search_url)
    response.raise_for_status()
    datasets = response.json()["result"]["results"]

    if not datasets:
        raise ValueError("No datasets found.")

    # Grab first dataset
    dataset = datasets[0]
    dataset_title = dataset["title"]
    resources = dataset["resources"]

    # Filter CSV resource
    csv_resources = [r for r in resources if "csv" in r["format"].lower()]
    if not csv_resources:
        raise ValueError("No CSV resources found.")

    csv_url = csv_resources[0]["url"]
    print(f"Dataset: {dataset_title}")
    print(f"Found resource URL: {csv_url}")
    return csv_url




In [4]:
# SECTION 3: Load Real-Time Grant Data (Dynamically from CKAN)
import requests
# SECTION 3: Load Real-Time Grant Data from CKAN & Preview Columns

try:
    csv_url = fetch_latest_csv_url_from_open_canada("grants contributions")
    grants_df = pd.read_csv(csv_url, encoding="ISO-8859-1", low_memory=False)
    print("Successfully fetched real-time grant data! Total entries:", len(grants_df))

    # Show column headers for review
    print("Column Headers:\n")
    for col in grants_df.columns:
        print("-", col)

except Exception as e:
    print("Failed to load dataset.", str(e))




Dataset: Proactive Disclosure - Grants and Contributions
Found resource URL: https://open.canada.ca/data/dataset/432527ab-7aac-45b5-81d6-7597107a7013/resource/1d15a62f-5656-49ad-8c88-f40ce689d831/download/grants.csv
Successfully fetched real-time grant data! Total entries: 1147926
Column Headers:

- ï»¿ref_number
- amendment_number
- amendment_date
- agreement_type
- recipient_type
- recipient_business_number
- recipient_legal_name
- recipient_operating_name
- research_organization_name
- recipient_country
- recipient_province
- recipient_city
- recipient_postal_code
- federal_riding_name_en
- federal_riding_name_fr
- federal_riding_number
- prog_name_en
- prog_name_fr
- prog_purpose_en
- prog_purpose_fr
- agreement_title_en
- agreement_title_fr
- agreement_number
- agreement_value
- foreign_currency_type
- foreign_currency_value
- agreement_start_date
- agreement_end_date
- coverage
- description_en
- description_fr
- naics_identifier
- expected_results_en
- expected_results_fr
- addi

In [5]:
try:
    # Combine the most relevant fields into a unified 'description' column
    description_sources = [
        "description_en",
        "prog_purpose_en",
        "expected_results_en",
        "additional_information_en",
        "agreement_title_en"
    ]

    description_parts = []

    for col in description_sources:
        if col in grants_df.columns:
            description_parts.append(grants_df[col].fillna(""))

    if not description_parts:
        raise ValueError("No usable columns found to generate grant description.")

    grants_df["description"] = description_parts[0]
    for part in description_parts[1:]:
        grants_df["description"] += " " + part

    # Rename columns to match internal schema
    grants_df.rename(columns={
        "recipient_legal_name": "name",
        "recipient_province": "region",
        "prog_name_en": "focus_area",
        "agreement_value": "funding_amount",
        "recipient_type": "recipient_type"
    }, inplace=True)

    # Filter for nonprofit organizations only
    grants_df = grants_df[grants_df["recipient_type"] == "N"]

    # Clean and format funding amounts
    grants_df["funding_amount"] = (
        grants_df["funding_amount"]
        .astype(str)
        .str.replace(r"[^\d.]", "", regex=True)
        .astype(float)
        .fillna(10000)
        .astype(int)
        .apply(lambda x: f"${x:,}")
    )

    # Drop rows with missing critical values
    grants_df.dropna(subset=["name", "region", "description", "focus_area"], inplace=True)

    # Drop duplicates
    grants_df.drop_duplicates(subset=["name", "description", "region"], inplace=True)

    print("Cleaned dataset is ready. Total filtered records:", len(grants_df))

except Exception as e:
    print("Failed to process dataset:", str(e))


Cleaned dataset is ready. Total filtered records: 158720


## Section 4: Analyze NGO Mission (Prompt 1)

This section uses Gemini to analyze the NGO's mission statement and extract structured metadata:
- Focus areas
- Target beneficiaries
- Keywords
- Operating region

These outputs are used in the next section to automatically score and match the best-fit grants from the real-time dataset.



In [6]:
# SECTION 4: Interactive NGO Input + Prompt 1

ngo_name_input = widgets.Text(
    placeholder='Enter NGO Name',
    description='NGO Name:',
    layout=widgets.Layout(width='70%')
)

ngo_mission_input = widgets.Textarea(
    placeholder='Enter NGO Mission...',
    description='NGO Mission:',
    layout=widgets.Layout(width='90%')
)

analyze_button = widgets.Button(description="Analyze Mission")
output_box = widgets.Output()

MISSION_ANALYSIS_PROMPT = """
You are a grant-matching assistant. Analyze the NGO's mission and return this JSON:
{
  "mission_summary": "...",
  "primary_focus_areas": [ "..." ],
  "target_beneficiaries": [ "..." ],
  "region": "...",
  "possible_grant_keywords": [ "..." ]
}
"""

def on_analyze_click(b):
    clear_output(wait=True)
    display(widgets.VBox([ngo_name_input, ngo_mission_input, analyze_button, output_box]))
    with output_box:
        ngo_name = ngo_name_input.value
        ngo_mission = ngo_mission_input.value
        print("Mission received. Analyzing with Gemini...\n")

        try:
            response = model.generate_content(
                MISSION_ANALYSIS_PROMPT + "\n\nNGO Mission:\n" + ngo_mission,
                generation_config=GenerationConfig(temperature=0.3)
            )
            cleaned_output = response.text.strip()
            if cleaned_output.startswith("```"):
                cleaned_output = "\n".join(cleaned_output.split("\n")[1:-1])
            global parsed_output
            parsed_output = json.loads(cleaned_output)
            print(json.dumps(parsed_output, indent=2))

        except Exception as e:
            print("Failed to parse output:")
            print(response.text)

analyze_button.on_click(on_analyze_click)

# Display UI
display(widgets.VBox([ngo_name_input, ngo_mission_input, analyze_button, output_box]))


In [ ]:
## Sample Prompt
# NGO Name: Future Skills Alberta
# We empower youth in rural Alberta through digital literacy and entrepreneurship training. Our programs help students build technical skills, gain confidence, and explore pathways to employment and self-employment. We focus on creating equitable access to future-ready education for underserved communities.


## Section 5: Match Top Grants from Dataset

In this section, we use the structured mission analysis from Gemini (Prompt 1) to identify and rank the most relevant grants from the real-time dataset.

To improve efficiency and scalability, the system uses a two-step process:

1. **Pre-filtering**  
   - Grants are first filtered based on regional alignment and keyword overlap in the description or focus area.
   - This reduces the candidate pool from thousands to only the most relevant grants.

2. **Gemini Scoring**  
   - The filtered grants are then evaluated using Gemini to assign a match score out of 5.
   - The score reflects how well each grant aligns with the NGO’s mission focus, region, and priorities.

The top 3 highest-scoring grants are selected for personalized proposal generation in the next step.




In [7]:
grants_df.head(10)[["name", "focus_area", "region", "funding_amount", "description"]]


,name,focus_area,region,funding_amount,description
1480,Vineland Research and Innovation Centre,AgriScience Program - Project Component,ON,"$1,172,910",The objective of this five-year project is to ...
1485,Canadian Food Exporters Association,AgriMarketing Program: National Industry Assoc...,ON,"$4,950,000",AgriMarketing Program - Small and Medium-sized...
1487,Prairie Oat Growers Association,AgriMarketing Program: National Industry Assoc...,SK,"$450,450",AgriMarketing Program - Small and Medium-sized...
1492,Canadian Thoroughbred Horse Society/Standardbr...,AgriAssurance Program: National Industry Assoc...,AB,"$511,018",The objective of this project is to enhance ex...
1495,Farm & Food Care Saskatchewan,AgriCommunication (ACOM),SK,"$296,604",The objective of this project will increase co...
1665,1426828 Ontario Inc.,AgriMarketing Program: Small and Medium-sized ...,ON,"$50,000",AgriMarketing Program - Small and Medium-sized...
2143,Canadian Cattlemen's Association,AgriScience Program - Project Component,AB,"$38,500",The objective of this project is to address th...
2144,Canadian Animal Health Coalition,AgriAssurance Program: National Industry Assoc...,ON,"$2,695,550",The objective of this project is to enhance An...
2147,Canadian Animal Health Coalition,AgriAssurance Program: National Industry Assoc...,ON,"$731,357",The project will develop and validate a real-t...
2148,Pulse Crops (Canada) Association,AgriAssurance Program: National Industry Assoc...,MB,"$1,601,846",The objective of this project is to generate t...


In [8]:
# SECTION 5A: Filter & Score Grants Based on Actual Dataset Structure

# Extract info from mission analysis
region_full = parsed_output["region"]          # e.g. "Alberta, Canada"
keywords = [kw.lower() for kw in parsed_output["possible_grant_keywords"]]

# Mapping of province names to codes
province_map = {
    "alberta": "AB", "british columbia": "BC", "manitoba": "MB",
    "new brunswick": "NB", "newfoundland": "NL", "nova scotia": "NS",
    "ontario": "ON", "prince edward island": "PE", "quebec": "QC",
    "saskatchewan": "SK", "northwest territories": "NT", "nunavut": "NU", "yukon": "YT"
}

# Extract province code from region
province_token = None
for name, code in province_map.items():
    if name in region_full.lower():
        province_token = code
        break

if province_token is None:
    print("Could not match province. Using fallback: no region filter.")
    province_token = ""

# Filter by province code
filtered_df = grants_df[
    grants_df["region"].str.upper().str.contains(province_token, na=False)
].copy()

# Filter by keyword presence in description only
def keyword_match(text):
    text = str(text).lower()
    return any(kw in text for kw in keywords)

filtered_df = filtered_df[
    filtered_df["description"].apply(keyword_match)
].copy()

# Drop rows with missing fields
filtered_df.dropna(subset=["name", "region", "description", "focus_area"], inplace=True)
filtered_df.reset_index(drop=True, inplace=True)

# Rule-based scoring
def rule_score(grant_row):
    try:
        region_val = str(grant_row.get("region", "")).upper()
        desc = str(grant_row.get("description", "")).lower()
        region_score = 1 if province_token in region_val else 0
        keyword_score = sum(1 for kw in keywords if kw in desc)
        return region_score + keyword_score
    except:
        return 0

filtered_df["pre_score"] = filtered_df.apply(rule_score, axis=1)

# Get top 10 candidates
top_10_candidates = filtered_df.sort_values(by="pre_score", ascending=False).head(10)
top_10_candidates.reset_index(drop=True, inplace=True)

# Display
display(top_10_candidates[["name", "focus_area", "region", "funding_amount", "pre_score"]])





,name,focus_area,region,funding_amount,pre_score
0,Town Of Stony Plain Public Library,Digital Literacy Exchange Program,AB,"$300,434",3
1,Chinook Arch Library Board,Digital Literacy Exchange Program,AB,"$438,800",3
2,Project Adult Literacy Society,Digital Literacy Exchange Program,AB,"$159,900",3
3,Parkland County Library Board,Digital Literacy Exchange Program,AB,"$195,641",3
4,Parkland County Library Board,Digital Literacy Exchange Program,AB,"$46,958",3
5,Society of the Lethbridge Community Network,Youth Internship Program,AB,"$421,705",2
6,Alberta Computers for Schools Association,Computers for Schools,AB,"$151,000",2
7,Alberta Computers for Schools Association,Computers for Schools Intern Program,AB,"$330,000",2
8,Society of the Lethbridge Community Network,Youth Internship Program,AB,"$421,705",2
9,Alberta Computers for Schools Association,Computers for Schools,AB,"$453,000",2


In [9]:
# SECTION 5B: Gemini scoring on top 10 filtered grants

def gemini_score(grant_row):
    prompt = f"""
    Evaluate how well this grant matches the NGO's mission profile.
    Return a score out of 5 only.

    NGO Region: {region_full}
    NGO Keywords: {', '.join(keywords)}

    Grant:
    Name: {grant_row['name']}
    Region: {grant_row['region']}
    Focus Area: {grant_row['focus_area']}
    Description: {grant_row['description']}
    """
    try:
        result = model.generate_content(prompt).text.strip()
        score = int(''.join(filter(str.isdigit, result)))
        return min(score, 5)
    except:
        return 0

top_10_candidates["match_score"] = top_10_candidates.apply(gemini_score, axis=1)

top_matches = top_10_candidates.sort_values(by="match_score", ascending=False).head(3)
top_matches_display = top_matches[["name", "focus_area", "region", "funding_amount", "match_score"]]
top_matches_display.reset_index(drop=True, inplace=True)

display(top_matches_display)


,name,focus_area,region,funding_amount,match_score
0,Town Of Stony Plain Public Library,Digital Literacy Exchange Program,AB,"$300,434",5
1,Chinook Arch Library Board,Digital Literacy Exchange Program,AB,"$438,800",5
2,Project Adult Literacy Society,Digital Literacy Exchange Program,AB,"$159,900",5


In [ ]:
# Now that we have the top three matches, we can choose any of the suitable matches and use Gemini to write up a proposal. 
# As an example, lets take the top Match Grant and use that for further processing.

## Section 6: Generate Tailored Proposal (Prompt 2)

In this step, we use Gemini to generate a personalized, funder-aligned proposal based on:

- The selected high-scoring grant (from Section 5B)
- The NGO’s mission statement (from Section 4)

The prompt combines these two inputs and instructs Gemini to generate a full grant proposal including:

- Title
- Summary (mentioning the NGO)
- 3 Objectives
- 2–3 Key Activities
- Impact Statement
- Requested Funding (aligned to the selected grant)

The result is a complete, context-aware proposal that NGOs can use as a starting point for real applications.


In [10]:
# SECTION 6: Generate Proposal using Gemini (Prompt 2)

grant_dropdown = widgets.Dropdown(
    options=list(top_matches["name"]),
    description="Select Grant:",
    layout=widgets.Layout(width='70%')
)

proposal_button = widgets.Button(description="📄 Generate Proposal")
proposal_output = widgets.Output()
full_proposal_text = ""

def on_generate_proposal(b):
    global full_proposal_text
    with proposal_output:
        clear_output(wait=True)

        selected_grant = grant_dropdown.value
        best_grant = top_matches[top_matches["name"] == selected_grant].iloc[0]

        grant_name = best_grant["name"]
        grant_description = best_grant["description"]
        grant_focus = best_grant["focus_area"]
        grant_funding = best_grant["funding_amount"]

        ngo_name = ngo_name_input.value
        ngo_mission = ngo_mission_input.value

        prompt = f"""
        You are a grant proposal writer. Based on the inputs below, write a tailored proposal.

        NGO Name: {ngo_name}
        NGO Mission: {ngo_mission}

        Grant Opportunity:
        - Name: {grant_name}
        - Description: {grant_description}
        - Focus Area: {grant_focus}
        - Funding: {grant_funding}

        Instructions:
        Write:
        - A project title
        - A summary (mention the NGO name)
        - 3 objectives
        - 2–3 main activities
        - An impact statement
        - Requested funding (same as grant)
        """

        response = model.generate_content(prompt, generation_config=GenerationConfig(temperature=0.7))
        full_proposal_text = response.text
        display(Markdown(full_proposal_text))

proposal_button.on_click(on_generate_proposal)

if not top_matches.empty:
    display(grant_dropdown, proposal_button, proposal_output)
else:
    print("No grants available to generate proposal.")


Dropdown(description='Select Grant:', layout=Layout(width='70%'), options=('Town Of Stony Plain Public Library…

Button(description='📄 Generate Proposal', style=ButtonStyle())

Output()

## Section 7: Structured Output (Prompt 3)

Here, we pass the generated proposal into Gemini again and ask it to return a structured JSON version for easier automation and integration.

In [11]:
generate_json_button = widgets.Button(description="Summarize Proposal as JSON")
json_output = widgets.Output()

@generate_json_button.on_click
def generate_json(b):
    global structured_json
    
    with json_output:
        clear_output(wait=True)

        if not full_proposal_text.strip():
            print("No proposal found. Please generate a proposal first.")
            return

        prompt = f"""
        You are a proposal analyzer. Extract the following structured JSON from the proposal:
        {{
          "proposal_title": "...",
          "summary": "...",
          "objectives": [ "..." ],
          "key_activities": [ "..." ],
          "impact_statement": "...",
          "requested_funding": "...",
          "timeline": "e.g., July to September 2025",
          "fit_score": "High / Medium / Low"
        }}

        Proposal:
        {full_proposal_text}
        """

        response = model.generate_content(prompt, generation_config=GenerationConfig(temperature=0.3))
        raw = response.text.strip()

        if raw.startswith("```"):
            raw = "\n".join(raw.split("\n")[1:-1])
        else:
            raw = raw.strip()

        try:
            structured_json = json.loads(raw)

            if not structured_json.get("proposal_title"):
                print("JSON seems incomplete. Try regenerating the proposal.")
                return

            display(Markdown("### Structured Proposal Summary"))
            display(Markdown("```json\n" + json.dumps(structured_json, indent=2) + "\n```"))

        except:
            print("JSON parsing failed. Raw output:")
            print(response.text)

display(generate_json_button, json_output)



Button(description='Summarize Proposal as JSON', style=ButtonStyle())

Output()

## Section 8: Final Output Summary

We now have a fully automated, GenAI-powered grant-matching and proposal generation workflow.

Here’s a recap of what this system achieved:

- Analyzed your NGO's mission and extracted key focus areas, keywords, and regions using **Prompt 1**
- Matched top grants from a real-time Canadian dataset using **hybrid filtering + Gemini scoring**
- Generated a personalized proposal using **Prompt 2**
- Converted that proposal into clean, structured JSON for automation or reuse using **Prompt 3**

---

### Reuse & Export

You can now:

- Copy and reuse the generated proposal in actual grant applications
- Export the structured JSON into grant submission portals
- Repeat the workflow with a different mission or new dataset!

---

*This system helps democratize access to funding for resource-limited nonprofits by combining smart filters, generative AI, and automation.*

---

## Final Grant Proposal (Formatted Output)

Section 9: Download Your Proposal

In [12]:
import json

# Save JSON to file
with open("structured_proposal_output.json", "w") as f:
    json.dump(structured_json, f, indent=2)

print("Structured proposal saved as structured_proposal_output.json")


Structured proposal saved as structured_proposal_output.json


In [13]:
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.set_font("Arial", size=12)

# Split into lines and add to PDF
for line in full_proposal_text.split("\n"):
    pdf.multi_cell(0, 10, txt=line.strip())

pdf.output("generated_proposal.pdf")
print("Full proposal saved as generated_proposal.pdf")


Full proposal saved as generated_proposal.pdf


In [14]:
!pip install python-docx
from docx import Document

# Create a new Word document
doc = Document()
doc.add_heading("Grant Proposal", level=1)

# Add NGO Name
doc.add_paragraph(f"NGO Name: {ngo_name_input.value}", style='Intense Quote')
doc.add_paragraph(" ")

# Add Proposal Content
for paragraph in full_proposal_text.strip().split("\n"):
    if paragraph.strip():  # Avoid blank lines
        doc.add_paragraph(paragraph.strip())

# Save the Word document
doc.save("generated_proposal.docx")
print("Word proposal saved as generated_proposal.docx")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 6.5 MB/s eta 0:00:00ta 0:00:01
Word proposal saved as generated_proposal.docx


In [15]:
from IPython.display import FileLink

display(FileLink("structured_proposal_output.json"))
display(FileLink("generated_proposal.pdf"))
display(FileLink("generated_proposal.docx"))


/kaggle/working/structured_proposal_output.json

/kaggle/working/generated_proposal.pdf

/kaggle/working/generated_proposal.docx